# 🔍 Fraud Model Comparison — Venn / Set Analysis

Compare the fraud detections of **3 ML models** side by side:
- Where does each model catch fraud **alone**?
- Where do they **overlap**?
- Where does **no model** catch fraud?

Each CSV/Excel/Parquet file should have:
- A **unique ID column** (to join the datasets)
- A **fraud label column** (1 = fraud, 0 = not fraud)

In [ ]:
# ── Install dependencies if needed ─────────────────────────────────────────
# !pip install matplotlib-venn upsetplot pandas openpyxl pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn3, venn3_circles
from upsetplot import UpSet, from_memberships
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded.')

## ⚙️ Configuration — Edit This Cell

In [ ]:
# ── FILE PATHS ─────────────────────────────────────────────────────────────
# Supported formats: .csv  |  .xlsx / .xls  |  .parquet

FILES = {
    'Model A': 'model_a.csv',
    'Model B': 'model_b.csv',
    'Model C': 'model_c.csv',
}

# ── COLUMN NAMES ───────────────────────────────────────────────────────────
ID_COL    = 'id'     # The unique identifier column (same name in all files)
LABEL_COL = 'fraud'  # The fraud label column: 1=fraud, 0=not fraud

# ── COLORS ─────────────────────────────────────────────────────────────────
COLORS = ['#E63946', '#2A9D8F', '#E9C46A']  # Red, Teal, Amber

## 📥 Load Datasets

In [ ]:
def load_file(path: str) -> pd.DataFrame:
    """Load CSV, Excel, or Parquet — auto-detected by extension."""
    ext = path.rsplit('.', 1)[-1].lower()
    loaders = {
        'csv':     pd.read_csv,
        'xlsx':    pd.read_excel,
        'xls':     pd.read_excel,
        'parquet': pd.read_parquet,
    }
    if ext not in loaders:
        raise ValueError(f'Unsupported file type: .{ext}')
    df = loaders[ext](path)
    assert ID_COL in df.columns,    f'Column "{ID_COL}" not found in {path}'
    assert LABEL_COL in df.columns, f'Column "{LABEL_COL}" not found in {path}'
    return df[[ID_COL, LABEL_COL]].copy()


dfs = {}
for name, path in FILES.items():
    dfs[name] = load_file(path)
    n_total = len(dfs[name])
    n_fraud = dfs[name][LABEL_COL].sum()
    print(f'  {name}: {n_total:,} rows | {int(n_fraud):,} fraud ({n_fraud/n_total:.1%})')

## 🔗 Build Merged Dataset

All records from all models are merged on the ID column. Each model gets its own flag column.

In [ ]:
model_names = list(dfs.keys())

# Rename label columns to model names
renamed = [
    df.rename(columns={LABEL_COL: name})
    for name, df in dfs.items()
]

# Outer join on ID — keeps every record seen in any model
merged = renamed[0]
for df in renamed[1:]:
    merged = merged.merge(df, on=ID_COL, how='outer')

# Fill NaN with 0 (record not present in that model's output → treated as not-fraud)
merged[model_names] = merged[model_names].fillna(0).astype(int)

print(f'Merged dataset: {len(merged):,} unique IDs')
merged.head()

## 📊 1 — Summary Table

In [ ]:
fraud_only = merged[merged[model_names].max(axis=1) == 1]  # at least one model flagged fraud

summary_rows = []
for name in model_names:
    detected = merged[name].sum()
    unique   = ((merged[name] == 1) & (merged[[n for n in model_names if n != name]].max(axis=1) == 0)).sum()
    summary_rows.append({'Model': name, 'Total Fraud Detected': int(detected), 'Unique (only this model)': int(unique)})

# All-three overlap
all_three = (merged[model_names].min(axis=1) == 1).sum()
any_model = (merged[model_names].max(axis=1) == 1).sum()

summary = pd.DataFrame(summary_rows)
print('=== Per-Model Summary ===')
print(summary.to_string(index=False))
print(f'\nCaught by ALL 3 models : {int(all_three):,}')
print(f'Caught by ANY model    : {int(any_model):,}')
print(f'Missed by ALL models   : {int((merged[model_names].max(axis=1) == 0).sum()):,}')

## 🟡 2 — Venn Diagram (Fraud Cases)

In [ ]:
A, B, C = [set(merged.loc[merged[n] == 1, ID_COL]) for n in model_names]

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#0F1117')

v = venn3([A, B, C], set_labels=model_names, ax=ax,
          set_colors=COLORS, alpha=0.55)

# Style labels
for text in v.set_labels:
    if text:
        text.set_color('white')
        text.set_fontsize(14)
        text.set_fontweight('bold')
for text in v.subset_labels:
    if text:
        text.set_color('white')
        text.set_fontsize(11)

# Circle outlines
c = venn3_circles([A, B, C], ax=ax, linewidth=1.5, color='white')

ax.set_title('Fraud Detection Overlap — Venn Diagram\n(counts = fraud cases)',
             color='white', fontsize=15, pad=18)

plt.tight_layout()
plt.savefig('venn_fraud.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved → venn_fraud.png')

## 📊 3 — UpSet Plot (more precise than Venn for 3+ sets)

In [ ]:
# Build membership list for all fraud-flagged records (union of all models)
fraud_rows = merged[merged[model_names].max(axis=1) == 1].copy()

memberships = [
    [name for name in model_names if row[name] == 1]
    for _, row in fraud_rows[model_names].iterrows()
]

upset_data = from_memberships(memberships)

fig = plt.figure(figsize=(12, 6))
fig.patch.set_facecolor('#0F1117')

upset = UpSet(upset_data, subset_size='count', show_counts=True,
              sort_by='cardinality',
              facecolor=COLORS[0], shading_color='#1E2130')

upset.plot(fig)
plt.suptitle('UpSet Plot — Fraud Detection Intersections', color='white',
             fontsize=14, y=1.02)

# Re-colour axes for dark background
for ax in fig.get_axes():
    ax.set_facecolor('#0F1117')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')

plt.savefig('upset_fraud.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved → upset_fraud.png')

## 📊 4 — Stacked Bar: Exclusive vs Shared Fraud

In [ ]:
def segment_counts(df, names):
    """Return counts of exclusive, partial, and full overlap fraud per model."""
    result = {}
    for name in names:
        others = [n for n in names if n != name]
        mask = df[name] == 1
        excl    = (mask & (df[others].max(axis=1) == 0)).sum()
        partial = (mask & (df[others].max(axis=1) == 1) & (df[others].min(axis=1) == 0)).sum()
        full    = (mask & (df[others].min(axis=1) == 1)).sum()
        result[name] = {'Exclusive': int(excl), 'Shared with ≥1': int(partial), 'All 3 agree': int(full)}
    return pd.DataFrame(result).T

seg = segment_counts(merged, model_names)

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#0F1117')

seg_colors = ['#E63946', '#F4A261', '#2A9D8F']
seg.plot(kind='bar', stacked=True, ax=ax, color=seg_colors, edgecolor='#0F1117', linewidth=0.8, width=0.55)

ax.set_title('Fraud Detections by Overlap Category', color='white', fontsize=14, pad=14)
ax.set_xlabel('Model', color='white', fontsize=11)
ax.set_ylabel('Fraud Cases', color='white', fontsize=11)
ax.tick_params(colors='white', rotation=0)
for spine in ax.spines.values():
    spine.set_edgecolor('#333')
ax.set_facecolor('#0F1117')
ax.grid(axis='y', color='#333', linestyle='--', alpha=0.5)

legend = ax.legend(title='Category', facecolor='#1E2130', edgecolor='#444',
                   labelcolor='white', title_fontsize=10)
legend.get_title().set_color('white')

# Add value labels
for bar_group in ax.containers:
    for bar in bar_group:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + h/2,
                    str(int(h)), ha='center', va='center',
                    color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('stacked_bar_fraud.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved → stacked_bar_fraud.png')

## 📊 5 — Heatmap of Pairwise Agreement

In [ ]:
import matplotlib.colors as mcolors

n = len(model_names)
agreement = np.zeros((n, n), dtype=int)

for i, a in enumerate(model_names):
    for j, b in enumerate(model_names):
        agreement[i, j] = ((merged[a] == 1) & (merged[b] == 1)).sum()

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#0F1117')

cmap = plt.cm.YlOrRd
im = ax.imshow(agreement, cmap=cmap, aspect='auto')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(model_names, color='white', fontsize=11)
ax.set_yticklabels(model_names, color='white', fontsize=11)
ax.tick_params(colors='white')

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{agreement[i,j]:,}',
                ha='center', va='center',
                color='black' if agreement[i,j] > agreement.max()*0.6 else 'white',
                fontsize=13, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(colors='white')

ax.set_title('Pairwise Fraud Agreement Heatmap\n(cells = cases both models flagged)',
             color='white', fontsize=12, pad=12)

for spine in ax.spines.values():
    spine.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('heatmap_agreement.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('Saved → heatmap_agreement.png')

## 📋 6 — Export Segment Breakdown to CSV

In [ ]:
# Add a 'segment' column to the merged dataset describing which combination flagged each record

def label_segment(row):
    flagged = [name for name in model_names if row[name] == 1]
    if len(flagged) == 0:
        return 'None'
    return ' ∩ '.join(flagged)

merged['segment'] = merged.apply(label_segment, axis=1)

segment_summary = (
    merged.groupby('segment')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

print(segment_summary.to_string(index=False))

merged.to_csv('fraud_merged_segments.csv', index=False)
segment_summary.to_csv('fraud_segment_summary.csv', index=False)
print('\nSaved → fraud_merged_segments.csv')
print('Saved → fraud_segment_summary.csv')